# 01 — Bronze Ingest (KaBuM) — ADLS JSONL → Delta (Unity Catalog)

Este notebook lê os arquivos **JSONL** do ADLS (camada *inbox*), cria um DataFrame Bronze e grava em **Delta Lake**, registrando a tabela no **Unity Catalog**.

**Fonte (ADLS / inbox):**  
`abfss://bronze@storagekabumdata.dfs.core.windows.net/inbox/kabum/bronze_scrape_v1/`

**Destino (Delta / bronze):**  
`abfss://bronze@storagekabumdata.dfs.core.windows.net/kabum/bronze/products_delta`

**Tabela UC:**  
`workspace.kabum_project.products_bronze`


In [0]:
%run ./00_config_uc


# 00 — Config (Unity Catalog + ADLS)

Centraliza catálogo/schema, storage account e paths base.


In [0]:
# ==========================================
# 0) CATALOG / SCHEMA
# ==========================================

spark.sql("USE CATALOG projeto_data_engineering")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS kabum_project
""")

spark.sql("USE SCHEMA kabum_project")

# ==========================================
# 1) PATHS
# ==========================================
BRONZE_SOURCE_PATH = "abfss://bronze@storagekabumdata.dfs.core.windows.net/inbox/kabum/bronze_scrape_v1/"
BRONZE_DELTA_PATH  = "abfss://bronze@storagekabumdata.dfs.core.windows.net/kabum/bronze/products_delta"

print("BRONZE_SOURCE_PATH:", BRONZE_SOURCE_PATH)
print("BRONZE_DELTA_PATH :", BRONZE_DELTA_PATH)


BRONZE_SOURCE_PATH: abfss://bronze@storagekabumdata.dfs.core.windows.net/inbox/kabum/bronze_scrape_v1/
BRONZE_DELTA_PATH : abfss://bronze@storagekabumdata.dfs.core.windows.net/kabum/bronze/products_delta


In [0]:
# ==========================================
# 2) VALIDANDO ARQUIVOS NA FONTE (JSONL)
# ==========================================
files = dbutils.fs.ls(BRONZE_SOURCE_PATH)
display(files)

# Checagem rápida: quantos arquivos e extensões
exts = {}
for f in files:
    name = f.name
    ext = name.split(".")[-1].lower() if "." in name else ""
    exts[ext] = exts.get(ext, 0) + 1
print("Arquivos:", len(files), "| Extensões:", exts)


In [0]:
# ==========================================
# 3) LER JSONL (cada linha = um JSON)
# ==========================================
from pyspark.sql import functions as F

df_bronze = (
    spark.read
         .format("json")
         .load(BRONZE_SOURCE_PATH)
         .withColumn("_src_file", F.expr("_metadata.file_path"))
         .withColumn("ingestion_date", F.current_date())
)

display(df_bronze.limit(50))


In [0]:
# ==========================================
# 4) VALIDAÇÃO RÁPIDA (schema + contagens)
# ==========================================
df_bronze.printSchema()
print("Rows    :", df_bronze.count())
print("Columns :", len(df_bronze.columns))

expected_cols = ["marketplace","search_term","product_url","product_name","price","brand"]
present = {c: (c in df_bronze.columns) for c in expected_cols}
print("Expected columns present:", present)


In [0]:
# ==========================================
# 5) GRAVAR BRONZE EM DELTA
# ==========================================

(
  df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .save(BRONZE_DELTA_PATH)
)

print("✅ Delta gravado em:", BRONZE_DELTA_PATH)


In [0]:
%sql
-- ==========================================
-- 6) REGISTRANDO TABELA NO UNITY CATALOG
-- ==========================================
CREATE TABLE IF NOT EXISTS projeto_data_engineering.kabum_project.products_bronze
USING DELTA
LOCATION 'abfss://bronze@storagekabumdata.dfs.core.windows.net/kabum/bronze/products_delta'


In [0]:
%sql
-- ==========================================
-- 7) VALIDANDO TABELA
-- ==========================================
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT product_url) AS distinct_products
FROM projeto_data_engineering.kabum_project.products_bronze;